In [ ]:
!pip install -U langchain
!pip install python-dotenv

import os
from dotenv import load_dotenv

# 바로 임포트
# os.environ["OPENAI_API_KEY"] ="sk-proj-IeiPd3N4ZP..."

# .env 파일 로드
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print(api_key)

In [ ]:
!pip install -U langchain ddgs wikipedia numexpr

In [ ]:
# !pip install -U langchain ddgs wikipedia numexpr

from langchain.agents import create_agent
from langchain.tools import tool
from ddgs import DDGS
import wikipedia, numexpr as ne

@tool("web_search", description="Search the web using DuckDuckGo.")
def web_search(q: str) -> str:
    with DDGS() as ddgs:
        res = ddgs.text(q, max_results=2)
        return "\n".join([f"{r['title']}: {r['body']} ({r['href']})" for r in res])

@tool("wiki", description="Retrieve summaries from Wikipedia.")
def wiki(q: str) -> str:
    try:
        return wikipedia.summary(q, sentences=2)
    except Exception as e:
        return str(e)

@tool("llm-math", description="Evaluate mathematical expressions safely.")
def calc(expr: str) -> str:
    try:
        return str(ne.evaluate(expr))
    except Exception as e:
        return f"Error: {e}"

agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search, wiki, calc],
    system_prompt="You are an assistant that can search the web, summarize Wikipedia, and calculate using math tools.",
)

response = agent.invoke({"messages": [{"role": "user", "content": "Find the population of Tokyo and divide it by 2."}]})
print(response["messages"][-1].content)
